# 02 — Preprocess EyePACS and OLIVES

One-time preprocessing into normalised 224×224 tensors saved to Drive.

**Run once only — resumable if it crashes.** EyePACS shards already on disk are skipped per-shard; the OLIVES output is skipped if its `.pt` file already exists.

Cells (b) and (c) shell out to `src.data.preprocess` as a CLI module so the same entry point works from the terminal.

## Session setup — clone repo, mount Drive, install requirements

In [ ]:
REPO_URL = "https://github.com/savita10/dr-dissertation.git"
REPO_DIR = "/content/dr-dissertation"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path(REPO_DIR).exists():
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

from src.utils.colab_setup import setup_colab

paths = setup_colab()

## Install PyTorch for Blackwell GPU (cu128)

Replaces the default torch wheel with the CUDA 12.8 build required for Blackwell-class GPUs (e.g. RTX 50-series, B200). Skip this cell if you are on a non-Blackwell runtime.

In [ ]:
!pip install --quiet --upgrade --index-url https://download.pytorch.org/whl/cu128 torch torchvision

import torch

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

## Show preprocess config

In [ ]:
from src.utils.config import load_config

preprocess_cfg = load_config("configs/preprocess.yaml")
print(preprocess_cfg)

## (a) Preprocess EyePACS only

Writes sharded `.pt` files plus `metadata.json` under `eyepacs_output`. Resumable: existing shards are skipped.

In [ ]:
!python -m src.data.preprocess --dataset eyepacs --config configs/preprocess.yaml

## (b) Preprocess OLIVES only

Extracts fundus members from the triple-nested `olive.zip` to `scratch_dir`, then writes a single `olives_fundus.pt` plus `metadata.json` under `olives_output`. Skipped entirely if the output file already exists.

In [ ]:
!python -m src.data.preprocess --dataset olives --config configs/preprocess.yaml

## (c) Verify both outputs

Loads one EyePACS shard and the OLIVES tensor file, prints shapes / dtypes / metadata.

In [ ]:
import json
from pathlib import Path

import torch

eyepacs_dir = Path(str(preprocess_cfg.eyepacs_output))
eyepacs_shards = sorted(eyepacs_dir.glob("eyepacs_shard_*.pt"))
print(f"EyePACS: {len(eyepacs_shards)} shards in {eyepacs_dir}")
if eyepacs_shards:
    shard = torch.load(eyepacs_shards[0], map_location="cpu", weights_only=False)
    print(f"  first shard: {eyepacs_shards[0].name}")
    print(f"    images: {tuple(shard['images'].shape)} {shard['images'].dtype}")
    print(f"    labels: {tuple(shard['labels'].shape)} {shard['labels'].dtype}")
    print(f"    filenames[:3]: {shard['filenames'][:3]}")
with open(eyepacs_dir / "metadata.json") as f:
    print("  metadata:", json.load(f))

olives_dir = Path(str(preprocess_cfg.olives_output))
olives_path = olives_dir / "olives_fundus.pt"
olives = torch.load(olives_path, map_location="cpu", weights_only=False)
print(f"\nOLIVES: {olives_path}")
print(f"  images: {tuple(olives['images'].shape)} {olives['images'].dtype}")
print(f"  labels: {tuple(olives['labels'].shape)} {olives['labels'].dtype}")
print(f"  biomarkers: {tuple(olives['biomarkers'].shape)} {olives['biomarkers'].dtype}")
print(f"  bcva: {tuple(olives['bcva'].shape)} {olives['bcva'].dtype}")
print(f"  cst: {tuple(olives['cst'].shape)} {olives['cst'].dtype}")
print(f"  metadata[:1]: {olives['metadata'][:1]}")
with open(olives_dir / "metadata.json") as f:
    print("  metadata.json:", json.load(f))